# Team findings: a common comparison

We compare our wheel-wear and profile-shape findings using the same truck assessments and held-out cars. Acceleration and displacement remain separate outcomes. Our questions are whether the models outperform simple controls and whether added measurements help consistently across cars.

This is exploratory analysis of a small, repeatedly studied sample. A higher score does not establish a physical cause. We retain negative and mixed results.

Run each cell from a fresh kernel. Original data stay unchanged; tables and figures are saved under `outputs/team_comparison/`.

**Setup:** keep this notebook in `notebooks/` and the supplied helper files in `src/`. Install `requirements.txt` plus XGBoost. Put the downloaded Teams Data contents in the repository's `data/` folder, or change the `inputs` line below to your own folder. Start Jupyter from the repository root. We provide the exact install and folder instructions with this notebook.


In [ ]:
from pathlib import Path
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Start Jupyter from the project folder or its notebooks folder.
root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
if not (root / "src/team_data.py").is_file():
    raise FileNotFoundError("Open this notebook from the project folder and include src/team_data.py.")
sys.path.insert(0, str(root / "src"))
from team_data import load_benchmark_data
from sklearn.base import clone
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import balanced_accuracy_score, recall_score, confusion_matrix
from xgboost import XGBClassifier

# Default: put the downloaded Data contents inside the project's data folder.
# If they are elsewhere, change this one line to Path("your data folder").
inputs = root / "data"
# Optional setting for an existing external data folder.
if os.environ.get("HUNTING_DATA_DIR"):
    inputs = Path(os.environ["HUNTING_DATA_DIR"]).expanduser()
    if not inputs.is_absolute():
        inputs = root / inputs
if not inputs.is_dir():
    raise FileNotFoundError("Data folder missing. Download the Teams Data folder and set inputs above.")
data = load_benchmark_data(inputs)
trucks = data["trucks"]
output = root / "outputs/team_comparison"
output.mkdir(parents=True, exist_ok=True)
display(pd.Series({"Cars": trucks.Car.nunique(), "Truck/rail assessments": len(trucks)}))


## Comparison rules

We hold out one complete car at a time, keeping its trucks and rail assessments together. We fix model settings before running the comparison; this first pass performs no tuning. Scaling is fitted inside each training fold. Rail identity uses the seven known categories; these tests concern unseen cars under existing rail conditions.

We first compare model families on the same measurements. We then compare Aaron's original and engineered feature sets. Additional shape comparisons use matching controls. Balanced accuracy, hunting recall and no-hunting recall are reported together.

The Gaussian-kernel comparisons retain the six-setting, three-inner-car-fold tuning used in our shape experiment. Other model settings below are fixed. These are pipeline comparisons; we do not claim equal optimization budgets or independently validated winners. [Nested validation reference](https://scikit-learn.org/stable/auto_examples/model_selection/plot_nested_cross_validation_iris.html).

We use Dom's eight-measurement averages/differences as the common feature block, adding conicity and rail identity and averaging axle features within a truck. Aaron's original/engineered RF features and settings are retained, but validation changes from truck-grouped folds to complete-car holdout. Jamie's contribution is the controlled shape/contact comparison; Sanskaar's is the descriptive rail-geometry comparison. These benchmark scores should not be substituted for the personal notebooks' original scores.

A positive outcome means onset below 80 mph. A value of 80 means no onset observed below the test limit; it is a censored observation, not proof that hunting cannot occur.


In [ ]:
rail = pd.get_dummies(trucks.Rail_Profile.astype(pd.CategoricalDtype(categories=range(1, 8))), prefix="rail", dtype=float)
models = {
    "Logistic regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, class_weight="balanced", C=1)),
    "Random forest": RandomForestClassifier(n_estimators=300, max_depth=4, min_samples_leaf=2, class_weight="balanced", random_state=42, n_jobs=1),
    "XGBoost": XGBClassifier(n_estimators=100, max_depth=2, learning_rate=.1, min_child_weight=3, random_state=42, n_jobs=1, eval_metric="logloss"),
}
experiments = [(name, "core", estimator) for name, estimator in models.items()]
experiments += [("RF original", "rf_original", models["Random forest"]),
                ("RF engineered", "rf_engineered", models["Random forest"]),
                ("LR: wear + conicity + rail", "wear", models["Logistic regression"]),
                ("LR: wear + shape + rail", "wear_shape", models["Logistic regression"])]
rows = []
for target in ["acceleration_hunted", "displacement_hunted"]:
    y = trucks[target].to_numpy(dtype=int)
    for name, features, estimator in experiments:
        X = np.column_stack([trucks[data["feature_sets"][features]].to_numpy(float), rail.to_numpy()])
        assert np.isfinite(X).all()
        for train, test in LeaveOneGroupOut().split(X, y, groups=trucks.Car):
            assert not set(trucks.Car.iloc[train]) & set(trucks.Car.iloc[test])
            fitted = clone(estimator).fit(X[train], y[train])
            predicted = fitted.predict(X[test]).astype(int)
            for index, prediction in zip(test, predicted):
                row = trucks.iloc[index]
                rows.append(dict(Car=int(row.Car), Truck=int(row.Truck), Rail_Profile=int(row.Rail_Profile), target=target, model=name, actual=int(y[index]), predicted=int(prediction)))
        print(target, name, "completed")
predictions = pd.DataFrame(rows)
assert not predictions.duplicated(["Car", "Truck", "Rail_Profile", "target", "model"]).any()
predictions.to_csv(output / "held_out_predictions.csv", index=False)


In [ ]:
from contact_hunting import evaluate_models
key = ["Car", "Truck", "Rail_Profile"]
kernel_features = {
    "KRR: conicity + rail": data["feature_sets"]["conicity"],
    "KRR: conicity + shape + rail": data["feature_sets"]["conicity_shape"],
    "KRR: wear + conicity + rail": data["feature_sets"]["wear"],
    "KRR: wear + shape + rail": data["feature_sets"]["wear_shape"],
    "KRR: wear + contact + rail": data["feature_sets"]["wear_contact"],
}
settings_frames = []
for target in ["acceleration_hunted", "displacement_hunted"]:
    _, held_out, settings = evaluate_models(trucks, target, feature_sets=kernel_features)
    assert not held_out.duplicated(key).any()
    assert held_out[key].equals(trucks[key])
    assert np.array_equal(held_out.actual, trucks[target])
    for model in ["Always hunting", "Rail only", *kernel_features]:
        frame = held_out[key + ["actual"]].copy()
        frame["target"], frame["model"], frame["predicted"] = target, model, held_out[model].to_numpy()
        predictions = pd.concat([predictions, frame], ignore_index=True)
    settings["target"] = target
    settings_frames.append(settings)
pd.concat(settings_frames, ignore_index=True).to_csv(output / "kernel_settings.csv", index=False)
assert not predictions.duplicated(key + ["target", "model"]).any()
predictions.to_csv(output / "held_out_predictions.csv", index=False)


## Results and per-car consistency

The table scores only held-out predictions. The error table keeps every car visible so a gain concentrated in one vehicle is easier to identify. These comparisons evaluate prespecified pipelines, including nested kernel tuning; they do not reproduce every teammate's original split or establish a winning algorithm.

In [ ]:
summary = []
for (target, model), group in predictions.groupby(["target", "model"]):
    tn, fp, fn, tp = confusion_matrix(group.actual, group.predicted, labels=[0, 1]).ravel()
    summary.append(dict(target=target, model=model, balanced_accuracy=balanced_accuracy_score(group.actual, group.predicted), hunting_recall=tp/(tp+fn), no_hunting_recall=tn/(tn+fp)))
metrics = pd.DataFrame(summary)
display(metrics.round(3))
metrics.to_csv(output / "metrics.csv", index=False)
errors = predictions.assign(error=lambda x: x.actual != x.predicted).groupby(["target", "Car", "model"]).error.sum().unstack()
display(errors)
errors.to_csv(output / "per_car_errors.csv")


## Paired checks and consolidated findings

We compare predictions for the same assessments and resample complete cars together. The ranges describe fixed held-out predictions, not retraining uncertainty or independent new-car validation. All planned comparisons are retained; these unadjusted exploratory ranges are not a multiple-testing significance claim.

In [ ]:
comparisons = [
    ("KRR: conicity + shape + rail", "KRR: conicity + rail", "Shape beyond conicity"),
    ("KRR: wear + shape + rail", "KRR: wear + conicity + rail", "Shape beyond standard wear"),
    ("KRR: wear + contact + rail", "KRR: wear + conicity + rail", "Contact beyond standard wear"),
    ("RF engineered", "RF original", "Aaron: engineered versus original"),
    ("Random forest", "Logistic regression", "Common features: RF versus logistic"),
    ("XGBoost", "Logistic regression", "Common features: XGBoost versus logistic"),
]
interval_rows = []
for target in predictions.target.unique():
    subset = predictions[predictions.target == target]
    for extended, baseline, question in comparisons:
        left = subset[subset.model == extended]
        right = subset[subset.model == baseline]
        joined = left.merge(right, on=key, validate="one_to_one", suffixes=("_new", "_base"))
        assert len(joined) == len(trucks)
        assert np.array_equal(joined.actual_new, joined.actual_base)
        cars = sorted(joined.Car.unique())
        counts = []
        for model_column in ["predicted_new", "predicted_base"]:
            counts.append(np.array([confusion_matrix(g.actual_new, g[model_column], labels=[0,1]).ravel()
                                    for _,g in joined.groupby("Car", sort=True)]))
        draws = np.random.default_rng(42).integers(0,len(cars),size=(2000,len(cars)))
        scores = []
        for c in counts:
            tn, fp, fn, tp = c[draws].sum(axis=1).T
            with np.errstate(divide="ignore",invalid="ignore"):
                scores.append(.5*(tn/(tn+fp)+tp/(tp+fn)))
        change = scores[0]-scores[1]
        change = change[np.isfinite(change)]
        assert len(change) > 0
        error_change = joined.assign(change=(joined.predicted_base != joined.actual_new).astype(int)
                                           -(joined.predicted_new != joined.actual_new).astype(int)).groupby("Car").change.sum()
        gain = balanced_accuracy_score(joined.actual_new, joined.predicted_new)-balanced_accuracy_score(joined.actual_new, joined.predicted_base)
        interval_rows.append(dict(target=target, question=question, gain_pp=100*gain,
            lower_pp=100*np.quantile(change,.025), upper_pp=100*np.quantile(change,.975),
            cars_improved=int((error_change>0).sum()),cars_worse=int((error_change<0).sum()),
            cars_tied=int((error_change==0).sum()),valid_resamples=len(change)))
intervals = pd.DataFrame(interval_rows)
display(intervals.round(2))
intervals.to_csv(output / "paired_comparisons.csv", index=False)


## Rail patterns and performance figures

The same trucks recur under seven rail conditions. Rail-level associations are descriptive and do not isolate a rail-shape cause. Complete-car validation tests new cars under these existing conditions, not unseen rail profiles.

In [ ]:
rail_results = trucks.groupby("Rail_Profile").agg(
    acceleration_hunting_rate=("acceleration_hunted","mean"),
    displacement_hunting_rate=("displacement_hunted","mean"),
    mean_conicity=("conicity_mean","mean"))
rail_results = rail_results.join(data["rail_summary"].set_index("Rail_Profile"))
display(rail_results.round(3))
rail_results.to_csv(output / "rail_findings.csv")
fig, axes = plt.subplots(1,2,figsize=(13,6),sharey=True)
for ax,target,title in zip(axes, ["acceleration_hunted","displacement_hunted"], ["Acceleration","Displacement"]):
    view = metrics[metrics.target==target].set_index("model")["balanced_accuracy"].reindex(metrics.model.drop_duplicates())
    view.plot.barh(ax=ax,color="#257C89")
    ax.axvline(.5,color="gray",linestyle="--",linewidth=1)
    ax.set(title=title,xlabel="Held-out balanced accuracy",ylabel="",xlim=(0,1))
fig.tight_layout()
fig.savefig(output / "model_comparison.png",dpi=300,bbox_inches="tight")
fig.savefig(output / "model_comparison.pdf",bbox_inches="tight")
plt.show()
fig, ax = plt.subplots(figsize=(10,5))
rail_results[["acceleration_hunting_rate","displacement_hunting_rate"]].plot.bar(ax=ax,rot=0)
ax.set(xlabel="Rail profile",ylabel="Fraction of truck assessments",title="Hunting rates by rail condition",ylim=(0,1))
fig.tight_layout();fig.savefig(output / "rail_hunting.png",dpi=300,bbox_inches="tight");fig.savefig(output / "rail_hunting.pdf",bbox_inches="tight");plt.show()


## What we can conclude

The findings table below keeps the original research questions separate from the evidence we obtained. A range crossing zero leaves the direction uncertain; a range on one side is an exploratory signal in this repeatedly studied sample, not proof of a general effect.

Brandon's published FIT/LSTM analysis uses pooled any-rail wheel labels, random wheel splits and augmentation before splitting. Its score answers a different question and can share related profiles across training and test. We retain the profile-learning idea, but its current score cannot support the shared truck/rail ranking. A car-grouped, correctly keyed FIT evaluation is required before a predictive conclusion can be drawn. We have not claimed a TensorFlow run in this notebook.

In [ ]:
findings = intervals.copy()
findings["interpretation"] = np.select(
    [findings.lower_pp>0, findings.upper_pp<0],
    ["Positive exploratory change; inspect per-car consistency", "Negative exploratory change under this pipeline"],
    default="Direction uncertain in this sample")
display(findings[["target","question","gain_pp","lower_pp","upper_pp","cars_improved","cars_worse","interpretation"]].round(2))
findings.to_csv(output / "consolidated_findings.csv", index=False)
import hashlib,json,platform,importlib.metadata
for path,digest in data["input_hashes"].items():
    assert hashlib.sha256(Path(path).read_bytes()).hexdigest() == digest, "Source input changed"
manifest = {"source_revision":json.loads((root/"SOURCE_REVISION.json").read_text())["main_revision"],
            "python":platform.python_version(),"seed":42,"bootstrap_draws":2000,
            "outer_split":"leave one complete car out", "inputs_unchanged":True,
            "packages":{name:importlib.metadata.version(name) for name in ["numpy","pandas","scikit-learn","xgboost"]},
            "input_hashes":data["input_hashes"],
            "code_sha256":{name:hashlib.sha256((root/name).read_bytes()).hexdigest() for name in ["notebooks/08_team_model_comparison.ipynb","src/team_data.py","src/axle_models.py","src/contact_hunting.py","src/profile_shapes.py","configs/comparison_contract.json"]},"limits":"Exploratory sample; fixed-prediction intervals omit retraining and feature-discovery uncertainty; no calibrated risk or causal claim."}
(output / "run_manifest.json").write_text(json.dumps(manifest,indent=2)+"\n")
print("Completed shared evaluation; outputs saved locally.")
